# Office-10 planted sampler fix
Runs only the 22-job planted suite in a new output directory. Keep completed Dirichlet results in the original project. A preflight checks actual class counts and both seed allocations before training.


In [ ]:
from pathlib import Path
import os, sys, subprocess
PROJECT = Path.cwd().resolve()
required = ['run_study.py','study.py','comparisons.py','summarize_study.py',
            'icde/config.py','icde/__init__.py','reference/run_domainnet10.py','legacy_protocol.json']
missing = [name for name in required if not (PROJECT/name).is_file()]
if missing and (PROJECT/'repair_sources.py').is_file():
    subprocess.run([sys.executable, str(PROJECT/'repair_sources.py')], cwd=PROJECT, check=True)
missing = [name for name in required if not (PROJECT/name).is_file()]
assert not missing, f'Incomplete project at {PROJECT}: {missing}. Extract the entire ZIP.'
ENV = dict(os.environ)
ENV['PYTHONPATH'] = str(PROJECT) + os.pathsep + ENV.get('PYTHONPATH','')
def run(*args):
    return subprocess.run([sys.executable, *map(str,args)], cwd=PROJECT, env=ENV, check=True)
print('Project:', PROJECT)


In [ ]:
DATASET = 'office10'  # domainnet10, digits5, office10
SUITE = 'planted'          # both, dirichlet, planted
DATA_ROOT = Path('/home/tahani/Paper4/ICDE_benchmark 6/data')
OUTPUT = PROJECT/'outputs'/f'{DATASET}_planted_capacity_v2'
SEEDS = [42, 123]
DEVICE = 'cuda'
WORKERS = 4
BATCH_SIZE = 64 if DATASET == 'digits5' else 32
AMP = True
RESUME = True
SAMPLES_PER_CLIENT = 500
REUSE_PLANTED = None    # Or Path('/path/to/old/planted/output/root')
PLANTED_SAMPLING = 'quota' if REUSE_PLANTED is not None else 'finite'

def arguments():
    args = ['run_study.py', '--dataset', DATASET, '--suite', SUITE,
            '--data-root', DATA_ROOT, '--output', OUTPUT, '--seeds', *SEEDS,
            '--device', DEVICE, '--workers', WORKERS, '--batch-size', BATCH_SIZE,
            '--samples-per-client', SAMPLES_PER_CLIENT, '--planted-sampling', PLANTED_SAMPLING]
    if AMP: args.append('--amp')
    if RESUME: args.append('--resume')
    if REUSE_PLANTED is not None: args.extend(['--reuse-planted', REUSE_PLANTED])
    return args
run(*arguments(), '--plan-only')


In [ ]:
import torch
print('PyTorch:', torch.__version__, 'CUDA available:', torch.cuda.is_available())
if DEVICE == 'cuda':
    assert torch.cuda.is_available(), 'Choose your CUDA-enabled Python kernel.'
    print('GPU:', torch.cuda.get_device_name(0))
assert (DATA_ROOT/DATASET).is_dir(), f'DATA_ROOT must contain {DATASET}; current path: {DATA_ROOT}'
run('check_data.py', '--data-root', DATA_ROOT, '--output', OUTPUT/'preflight')
run('-m', 'unittest', 'discover', '-s', 'tests', '-v')
run(*arguments())


In [ ]:
run('summarize_study.py', '--output', OUTPUT)
for name in ['summary.csv','paired_differences.csv','communication_per_run.csv']:
    path = OUTPUT/name
    if path.exists(): print(name, '\n', path.read_text())
    else: print('No completed results yet:', path)


Run the final check only after all requested jobs finish. Return the three CSVs above, `per_run.csv`, and the `audits` folder. Each grouped run also contains its neighbor details and matched-checkpoint hashes. Missing planted metrics in the Dirichlet suite are intentional.

In [ ]:
run('verify_outputs.py', '--output', OUTPUT)
